In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

## Data Deinition

#### bank_customers

"customer_id": "Unique customer identifier (format: CUST-XXXXXXX)"

"contact_timestamp": "Date and time when the customer was last contacted during the campaign",

"age": "Customer's age in years (range: 18-95)",

"gender": "Customer's gender (Male / Female / Other)",

"marital_status": "Marital status (Single / Married / Divorced / Widowed)",

"education": "Highest education level (High School / Some College / University Degree / Professional Degree / Postgraduate)",

"occupation": "Current occupation category (Blue-Collar / Services / Technician / Admin / Management / Entrepreneur / Self-Employed / 
Student / Retired / Unemployed)",

"city": "City of residence (18 major Indian cities)",

"annual_income": "Annual income in INR (range: 50,000 - 15,000,000). Correlated with education, occupation, and age",

"account_balance": "Current account balance in INR. Negative values indicate overdraft (~8% of customers). Correlated with income",

"credit_score": "Credit score (range: 300-900). Correlated with income and age",

"customer_tenure_years": "Number of years as a bank customer. Upper-bounded by age minus 18",

"has_mortgage": "Whether the customer has a mortgage with the bank (0 = No, 1 = Yes). Higher likelihood for age 28-60 with income > 400K",

"has_personal_loan": "Whether the customer has an active personal loan (0 = No, 1 = Yes)",

"has_credit_card": "Whether the customer holds a credit card (0 = No, 1 = Yes). Higher likelihood for income > 300K",

"num_products": "Total number of banking products held by the customer (range: 1-7)",

"contact_method": "Method used to contact the customer in this campaign (Cellular / Telephone / Email)",

"campaign_month": "Month when the customer was contacted (full month name)",

"day_of_week": "Day of the week when the customer was contacted (full day name)",

"contacts_this_campaign": "Number of times the customer was contacted during the current campaign (range: 1-19, right-skewed)",

"days_since_prev_contact": "Number of days since last contact from a previous campaign. -1 indicates the customer was never previously 
contacted",

"contacts_prev_campaigns": "Total number of contacts made to the customer in all previous campaigns. 0 if never previously contacted",

"prev_campaign_outcome": "Outcome of the previous marketing campaign (Success / Failure / Nonexistent). Nonexistent if no prior campaign 
contact",

"has_online_banking": "Whether the customer uses online banking (0 = No, 1 = Yes). Higher adoption for age < 50",

"has_mobile_app": "Whether the customer uses the bank's mobile app (0 = No, 1 = Yes). Strongly correlated with online banking adoption",

"email_open_rate": "Historical email open rate for marketing emails (range: 0.0 - 1.0)",

"email_click_rate": "Historical email click-through rate (range: 0.0 - 1.0). Always <= email_open_rate",

"website_visits_last_30d": "Number of visits to the bank's website in the last 30 days. Higher for digital-active customers",

"emp_variation_rate": "Quarterly employment variation rate indicator (macroeconomic feature)",

"consumer_price_index": "Monthly consumer price index at the time of contact (macroeconomic feature)",

"consumer_confidence_index": "Monthly consumer confidence index at the time of contact (macroeconomic feature, negative scale)",

"euribor_3m_rate": "Euribor 3-month rate at the time of contact (macroeconomic feature)",

"nr_employed": "Number of employees in the economy (quarterly indicator, in thousands)",

"responded": "TARGET VARIABLE. Whether the customer responded positively to the campaign (0 = No, 1 = Yes). ~12.5% positive rate"

#### bank_transactions
"transaction_id": "Unique transaction identifier (format: TXN-XXXXXXXXXX)",

"customer_id": "Foreign key linking to bank_customers table",

"transaction_date": "Date and time of the transaction. All transactions fall within 12 months before the customer's campaign contact date",

"transaction_type": "Type of transaction (Debit / Credit / Transfer / ATM Withdrawal / POS Purchase / Online Purchase / Bill Payment / EMI 
Payment / UPI Transfer / Refund)",

"amount": "Transaction amount in INR. Correlated with customer's annual income. Scale varies by transaction type",

"channel": "Channel through which the transaction was executed (Branch / ATM / Online Banking / Mobile App / POS Terminal / UPI). Digital 
customers skew towards online and mobile channels",


"merchant_category": "Category of the merchant or purpose (Groceries / Fuel / Restaurants / Utilities / Healthcare / Education / 
Entertainment / Travel / Shopping / Insurance / Rent / Investments / Salary Credit / Loan Disbursement / Other). Mapped logically to 
transaction type",

"balance_after_txn": "Running account balance after the transaction was processed. Credits increase balance, debits decrease it",

"is_flagged": "Whether the transaction was flagged for review (0 = No, 1 = Yes). ~1.2% base rate, ~8% for transactions above 95th percentile amount"


In [0]:
# # Credit Card Transactions Data
# credit_card_transactions = spark.read.csv(
#     "/Volumes/aidetic_databricks/default/credit_card_transactions/credit_card_transactions.csv",
#     header=True,
#     inferSchema=True
# )
# credit_card_transactions = credit_card_transactions.drop("Unnamed: 0")

# # Bank Customer Data
# credit_card_transactions = spark.read.csv(
#     "/Volumes/aidetic_databricks/default/credit_card_transactions/bank_customers.csv",
#     header=True,
#     inferSchema=True
# )

# Bank Transaction Data
credit_card_transactions = spark.read.csv(
    "/Volumes/aidetic_databricks/default/credit_card_transactions/bank_transactions.csv",
    header=True,
    inferSchema=True
)

display(credit_card_transactions.limit(5))

In [0]:
for c in credit_card_transactions.columns:
    credit_card_transactions = credit_card_transactions.withColumnRenamed(c, c.replace(" ", "_"))
    credit_card_transactions = credit_card_transactions.withColumnRenamed(c, c.replace(".", "_"))

In [0]:
display(credit_card_transactions.head(5))

In [0]:
CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"

# # Bank Customers
# TABLE_NAME = "hdfc_demo_bank_customers"

# Bank Customers
TABLE_NAME = "hdfc_demo_bank_transactions"

# credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"
credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

In [0]:
# Write to tables in Unity Catalog
spark.sql(f"DROP TABLE IF EXISTS {credit_card_transactions_table_name}")
credit_card_transactions.write.saveAsTable(credit_card_transactions_table_name)
